# Notebook 02: Mô Phỏng Cơ Chế MPIN (Example 1 Paper Demo)

**Mục tiêu:** Tái hiện lại ví dụ minh họa cốt lõi trong **Section 3.2 (Example 1)** của paper MPIN:
- So sánh giữa **FeaProp** (trọng số cố định) và **MsgProp** (trọng số tương quan học được).
- Minh họa vì sao việc học trọng số tương quan mang lại kết quả điền chính xác hơn.

In [ ]:
import sys
import os
import numpy as np
import torch
import torch.nn as nn

sys.path.append(os.path.abspath("../src"))
from models.similarity_graph import build_similarity_graph
from models.baselines import feature_propagation

## 1. Dữ Liệu Ví Dụ 1 Trong Paper
- Node $x_2 = [\text{null}, 5, 6]$ có ground truth tại vị trí thiếu là **12**.
- Hàng xóm $x_3 = [8, 5, 3]$
- Hàng xóm $x_4 = [3, 2, 1]$

In [ ]:
x2_true = np.array([12.0, 5.0, 6.0])
x2 = np.array([np.nan, 5.0, 6.0])
x3 = np.array([8.0, 5.0, 3.0])
x4 = np.array([3.0, 2.0, 1.0])

print(f"Node x2 (cần điền): {x2}")
print(f"Ground Truth của x2[0]: {x2_true[0]}")

## 2. Tính Toán với Feature Propagation (FeaProp)
FeaProp chia đều trọng số cho các hàng xóm:
$$\tilde{x}_2[0] = 0.5 \cdot x_3[0] + 0.5 \cdot x_4[0] = 0.5 \cdot 8 + 0.5 \cdot 3 = 5.5$$

In [ ]:
feaprop_imputed = 0.5 * x3[0] + 0.5 * x4[0]
feaprop_error = abs(feaprop_imputed - x2_true[0])
print(f"Giá trị FeaProp điền: {feaprop_imputed:.2f}")
print(f"Sai số tuyệt đối (MAE) của FeaProp: {feaprop_error:.2f}")

## 3. Tính Toán với Message Propagation (MsgProp)
Vì $x_3$ có các đặc trưng tương đồng với $x_2$ hơn $x_4$, cơ chế học tương quan gán trọng số cao hơn cho $x_3$ (ví dụ: 0.8 cho $x_3$ và 0.2 cho $x_4$):
$$\tilde{x}_2[0] = 0.8 \cdot 8 + 0.2 \cdot 3 = 6.4 + 0.6 = 7.0$$

In [ ]:
msgprop_imputed = 0.8 * x3[0] + 0.2 * x4[0]
msgprop_error = abs(msgprop_imputed - x2_true[0])
print(f"Giá trị MsgProp điền: {msgprop_imputed:.2f}")
print(f"Sai số tuyệt đối (MAE) của MsgProp: {msgprop_error:.2f}")
print(f"-> MsgProp (7.0) gần ground truth (12.0) hơn đáng kể so với FeaProp (5.5)!")